<a href="https://colab.research.google.com/github/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_4/lessons/lesson_40_auth_security/note_lesson_40_auth.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔐 Урок 40 — Автентифікація й безпека: рефакторинг 4 застосунку нотаток

| Крок | Що робимо |
|---|---|
| 0 | `crispy_notes_project` з групами, скиданням пароля і JWT у ноутбуці |
| 1 | нотатки групи: хто бачить і хто змінює (вправи 1–2) |
| 2 | паролі: хеш, сіль, перевірка (вправи 3–4) |
| 3 | JWT: видати, прочитати, використати, підправити, оновити (вправи 5–8) |
| 4 | перебір паролів і throttle (вправа 9) |
| 5 | правило автора в одному місці — DRF-permission (вправа 10) |

**Як працювати:** зверху вниз; перед **🔮 Прогнозом** спершу відповідай сам. Зміни коду, знахідки й архітектура — у книзі курсу: [Урок 40](https://nikoriakviktot.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m4/lesson_40/); теорія — частина VII Django-книги [Auth і безпека](https://nikoriakviktot.github.io/notes_chat_app/07_auth_and_security/).

---
## 0. Проєкт у ноутбуці

`crispy_notes_project` з уроку 35 (сторінки + DRF API) + код уроку автентифікації старого курсу (групи, скидання пароля, налаштування безпеки) + JWT (`/api/token/`). У Colab клітинка нижче завантажить лише папку уроку й встановить залежності.

In [ ]:
import os
import subprocess
import sys

LESSON = "module_4/lessons/lesson_40_auth_security"
REPO = "https://github.com/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026.git"

if not os.path.isdir("crispy_notes_project"):
    if not os.path.isdir("course_repo"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--filter=blob:none", "--sparse", REPO, "course_repo"], check=True)
        subprocess.run(["git", "-C", "course_repo", "sparse-checkout", "set", LESSON], check=True)
    os.chdir(os.path.join("course_repo", LESSON))

try:
    import crispy_bootstrap5, debug_toolbar, drf_spectacular, rest_framework, rest_framework_simplejwt  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "crispy_notes_project/requirements.txt"], check=True)

os.chdir("crispy_notes_project")
if os.path.exists("db.sqlite3"):
    os.remove("db.sqlite3")                             # щоразу — чиста база
result = subprocess.run([sys.executable, "manage.py", "migrate"], capture_output=True, text=True)
print("\n".join((result.stdout or result.stderr).splitlines()[-2:]))

In [ ]:
import base64
import json

import django

sys.path.insert(0, os.getcwd())
os.environ.setdefault("DJANGO_SETTINGS_MODULE", "hello_project.settings")
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"      # Jupyter працює в asyncio-циклі (урок 33)
django.setup()

from django.contrib.auth.hashers import check_password
from django.contrib.auth.models import User
from django.core.cache import cache
from django.test.utils import setup_test_environment
from rest_framework.test import APIClient

from hello_app import selectors, services

setup_test_environment()                               # хост testserver для тестового клієнта
olena = User.objects.create_user("olena", password="Sup3r-secret!")
taras = User.objects.create_user("taras", password="Sup3r-secret!")
ivan = User.objects.create_user("ivan", password="Sup3r-secret!")
wifi = services.create_note(user=olena, title="Пароль від Wi-Fi")
print("користувачі:", [u.username for u in User.objects.all()], "| нотатка:", wifi.title)

---
## 1. Нотатки групи

### Вправа 1

Створи групу «Сім'я» з Оленою (`services.create_group`), додай Тараса (`group.user_set.add`) і зроби нотатку `wifi` нотаткою групи (`services.update_note(wifi, group=family)`). Потім збери, хто з трьох користувачів бачить її у своєму списку (`selectors.get_user_notes`).

In [ ]:
# YOUR CODE HERE
# BEGIN SOLUTION
family = services.create_group(name="Сім'я", creator=olena)
family.user_set.add(taras)
wifi = services.update_note(wifi, group=family)
sees = [user.username for user in (olena, taras, ivan) if wifi in selectors.get_user_notes(user)]
# END SOLUTION

print(sees)
assert sees == ["olena", "taras"]
print("✅ Вправа 1 пройдена")

**🔮 Прогноз:** Тарас (у групі) і Іван (ні) звертаються до API нотатки групи. Які статуси: Тарас `GET`, Тарас `PATCH`, Тарас `DELETE`, Іван `GET`?

<details>
<summary>Відповідь</summary>

`200`, `403`, `403`, `404`. Тарас бачить нотатку, але змінювати може лише автор — чесне «заборонено». Іван нотатку не бачить — для нього її «немає».

</details>

### Вправа 2

Перевір прогноз: збери статуси в словник `codes` з ключами `"taras GET"`, `"taras PATCH"`, `"taras DELETE"`, `"ivan GET"` (`APIClient` і `force_authenticate`).

In [ ]:
api = APIClient()
url = f"/api/notes/{wifi.pk}/"

# YOUR CODE HERE
# BEGIN SOLUTION
codes = {}
api.force_authenticate(taras)
codes["taras GET"] = api.get(url).status_code
codes["taras PATCH"] = api.patch(url, {"title": "Зламано"}, format="json").status_code
codes["taras DELETE"] = api.delete(url).status_code
api.force_authenticate(ivan)
codes["ivan GET"] = api.get(url).status_code
# END SOLUTION

print(codes)
assert codes == {"taras GET": 200, "taras PATCH": 403, "taras DELETE": 403, "ivan GET": 404}
print("✅ Вправа 2 пройдена")

---
## 2. Паролі

У `user.password` — не пароль, а рядок `алгоритм$ітерації$сіль$хеш`.

### Вправа 3

Розбери `olena.password` і `taras.password` (однаковий пароль!) через `split("$")`: алгоритм — у `algorithm`, число ітерацій (`int`) — у `iterations`, а `same_hash` — чи однакові частини-хеші в обох.

In [ ]:
# YOUR CODE HERE
# BEGIN SOLUTION
algorithm, iterations, salt, digest = olena.password.split("$")
iterations = int(iterations)
same_hash = digest == taras.password.split("$")[3]
# END SOLUTION

print(algorithm, iterations, "| той самий хеш?", same_hash)
assert algorithm == "pbkdf2_sha256" and iterations >= 600_000 and same_hash is False
print("✅ Вправа 3 пройдена")

**🔮 Прогноз:** Чому однаковий пароль дав різні хеші? І як тоді сервер перевіряє пароль при вході?

<details>
<summary>Відповідь</summary>

Сіль — випадкова для кожного користувача, і хешується пароль разом із сіллю. При вході сервер бере сіль з рядка в базі, рахує хеш введеного пароля з нею й порівнює — `check_password`.

</details>

### Вправа 4

Перевір `check_password` для правильного пароля і для пароля з іншою першою літерою (`sup3r-secret!`). Результати — у `right` і `wrong`.

In [ ]:
# YOUR CODE HERE
# BEGIN SOLUTION
right = check_password("Sup3r-secret!", olena.password)
wrong = check_password("sup3r-secret!", olena.password)
# END SOLUTION

print(right, wrong)
assert (right, wrong) == (True, False)
print("✅ Вправа 4 пройдена")

---
## 3. JWT

`POST /api/token/` з логіном і паролем → `{"access": …, "refresh": …}`. Токен — три частини через крапку: `header.payload.signature`, кожна в base64url.

### Вправа 5

Отримай токени Олени (у `tokens`) і прочитай payload access-токена **без ключа**: друга частина, base64url з доповненням `=` до довжини, кратної 4, потім `json.loads`. Результат — у `claims`.

In [ ]:
cache.clear()                                           # лічильник спроб входу — у кеші
api = APIClient()

# YOUR CODE HERE
# BEGIN SOLUTION
tokens = api.post("/api/token/", {"username": "olena", "password": "Sup3r-secret!"}, format="json").json()
payload_part = tokens["access"].split(".")[1]
claims = json.loads(base64.urlsafe_b64decode(payload_part + "=" * (-len(payload_part) % 4)))
# END SOLUTION

print({key: claims[key] for key in ("token_type", "user_id")}, "| живе", claims["exp"] - claims["iat"], "с")
assert claims["user_id"] == str(olena.id) and claims["token_type"] == "access"
print("✅ Вправа 5 пройдена")

### Вправа 6

Надішли `GET /api/notes/` без токена (статус — у `anonymous`) і з заголовком `Authorization: Bearer <access>` (`api.credentials(HTTP_AUTHORIZATION=...)`, назви нотаток — у `titles`).

In [ ]:
# YOUR CODE HERE
# BEGIN SOLUTION
anonymous = api.get("/api/notes/").status_code
api.credentials(HTTP_AUTHORIZATION=f"Bearer {tokens['access']}")
titles = [note["title"] for note in api.get("/api/notes/").json()]
# END SOLUTION

print(anonymous, titles)
assert anonymous == 401 and titles == ["Пароль від Wi-Fi"]
print("✅ Вправа 6 пройдена")

**🔮 Прогноз:** Тарас скопіював токен Олени, у payload замінив `user_id` на свій і закодував назад. Підпис лишив Оленин. Що відповість сервер?

<details>
<summary>Відповідь</summary>

`401`: підпис рахувався від **старих** header і payload. Сервер перераховує HMAC від нових — і він не збігається. Без `SECRET_KEY` правильний підпис для зміненого payload не зробити.

</details>

### Вправа 7

Перевір прогноз: зроби `forged_payload` — той самий payload, але `user_id` = id Тараса (JSON → base64url без `=`), склей `header.forged_payload.signature` і надішли запит. Статус — у `status_forged`.

In [ ]:
header, payload_part, signature = tokens["access"].split(".")

# YOUR CODE HERE
# BEGIN SOLUTION
forged_claims = {**claims, "user_id": str(taras.id)}
forged_payload = base64.urlsafe_b64encode(json.dumps(forged_claims).encode()).decode().rstrip("=")
api.credentials(HTTP_AUTHORIZATION=f"Bearer {header}.{forged_payload}.{signature}")
status_forged = api.get("/api/notes/").status_code
# END SOLUTION

print(status_forged, api.get("/api/notes/").json()["detail"])
assert status_forged == 401
print("✅ Вправа 7 пройдена")

### Вправа 8

Обміняй refresh-токен на нову пару (`POST /api/token/refresh/`). Збережи відповідь у `fresh` і перевір, що новий access працює, а refresh змінився (ротація).

In [ ]:
api.credentials()

# YOUR CODE HERE
# BEGIN SOLUTION
fresh = api.post("/api/token/refresh/", {"refresh": tokens["refresh"]}, format="json").json()
api.credentials(HTTP_AUTHORIZATION=f"Bearer {fresh['access']}")
works = api.get("/api/notes/").status_code == 200
# END SOLUTION

print(sorted(fresh), "| працює:", works, "| новий refresh:", fresh["refresh"] != tokens["refresh"])
assert works and fresh["refresh"] != tokens["refresh"]
print("✅ Вправа 8 пройдена")

---
## 4. Перебір паролів

`/api/token/` пускає не більше 5 спроб за хвилину з однієї адреси (`ScopedRateThrottle`).

### Вправа 9

Скинь кеш і надсилай неправильні паролі, поки не отримаєш `429`. Статуси — у `attempts`; потім спробуй **правильний** пароль (статус — у `after`).

In [ ]:
cache.clear()
api = APIClient()

# YOUR CODE HERE
# BEGIN SOLUTION
attempts = []
while not attempts or attempts[-1] != 429:
    attempts.append(api.post("/api/token/", {"username": "olena", "password": "guess"}, format="json").status_code)
after = api.post("/api/token/", {"username": "olena", "password": "Sup3r-secret!"}, format="json").status_code
# END SOLUTION

print(attempts, "| правильний пароль:", after)
assert attempts == [401] * 5 + [429] and after == 429
print("✅ Вправа 9 пройдена")

---
## 5. Правило автора в одному місці

Знахідка уроку: сайт перевіряв автора, а API — ні. Правило, записане в кількох місцях, колись забудуть. У DRF для цього — permission-клас з `has_object_permission(request, view, obj)`.

### Вправа 10

Напиши `IsAuthorOrReadOnly`: безпечні методи (`permissions.SAFE_METHODS`: `GET`, `HEAD`, `OPTIONS`) — дозволено; решта — лише якщо `obj.user_id == request.user.id`.

In [ ]:
from types import SimpleNamespace

from rest_framework import permissions


# YOUR CODE HERE
# BEGIN SOLUTION
class IsAuthorOrReadOnly(permissions.BasePermission):
    message = "Змінювати й видаляти нотатку може лише її автор."

    def has_object_permission(self, request, view, obj):
        if request.method in permissions.SAFE_METHODS:
            return True
        return obj.user_id == request.user.id
# END SOLUTION

check = IsAuthorOrReadOnly()
ask = lambda method, user: check.has_object_permission(SimpleNamespace(method=method, user=user), None, wifi)
table = {(method, user.username): ask(method, user) for method in ("GET", "PATCH", "DELETE") for user in (olena, taras)}
print(table)
assert table == {("GET", "olena"): True, ("GET", "taras"): True, ("PATCH", "olena"): True,
                 ("PATCH", "taras"): False, ("DELETE", "olena"): True, ("DELETE", "taras"): False}
print("✅ Вправа 10 пройдена")

---
## Самоперевірка

1. Чим автентифікація відрізняється від авторизації? Що з них дає JWT?
2. Чому учасник групи отримує `403`, а сторонній — `404`?
3. Навіщо сіль і навіщо хешування навмисно повільне?
4. Що в JWT можна прочитати без ключа і чого без ключа зробити не можна?
5. Чому `SECRET_KEY` не можна тримати в публічному репозиторії?

<details>
<summary>Відповіді</summary>

1. Автентифікація — «хто ти», авторизація — «що тобі можна». JWT (як і сесія) — лише автентифікація; права вирішує код.
2. Учасник і так бачить нотатку — приховувати нічого; сторонньому `403` підтвердив би, що нотатка існує.
3. Сіль робить однакові паролі різними хешами й ламає заздалегідь пораховані таблиці; повільність робить перебір після витоку бази надто довгим.
4. Прочитати header і payload (base64); змінити їх і мати правильний підпис — ні.
5. Ним підписано JWT, сесії й посилання скидання пароля: хто знає ключ, сам випустить «справжній» токен для будь-якого користувача.

</details>

## Далі

- Книга курсу: [Урок 40](https://nikoriakviktot.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m4/lesson_40/) — знахідка «API обходив перевірку автора», скидання пароля, `check --deploy`, OWASP Top 10 на нашому коді, «Знайди помилку» з `SECRET_KEY`.
- У папці `crispy_notes_project`: `python manage.py test`, `python manage.py check --deploy`.
- **Урок 41** — тестування API агрегатора новин.